# Map pUC19

Highlight the multiple cloning site on pUC19 and render the circular map used in the README. The circular and linear maps use the desktop’s SVG renderer. The coordinate overview uses the same layout as the desktop overview. [Annotated plasmid](../../databases/plasmids/maps/pUC19.gb) · [Map labels and coordinates](../../databases/plasmids/README.md#puc19-map-labels)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "strandfolio" / "__init__.py").is_file())
sys.path.insert(0, str(root))

from IPython.display import SVG, display

from strandfolio.application.session import DocumentSession
from strandfolio.io.formats import readPastedText, readSequence
from strandfolio.operations.conversion import GeneticCodePreference, translateFrame
from strandfolio.rendering.maps import linearMapSvg, overviewSvg, plasmidMapSvg

## Select the multiple cloning site

Look up the feature by its label and let the session select its annotated interval. The 57-base polylinker spans bases 233–289 in the deposited sequence.

In [ ]:
plasmid = readSequence(root / "databases/plasmids/maps/pUC19.gb").records[0]
session = DocumentSession(plasmid)
mcs = next(feature for feature in plasmid.features if feature.label == "MCS")
session.selectWholeFeature(mcs.featureId)

print(f"{plasmid.name} · {len(plasmid):,} bp · {mcs.label} selected ({len(session.selection)} bp)")
circular = plasmidMapSvg(session, width=960, height=640)
display(SVG(circular))

## Compare views of the selection

The linear map lays out the annotated features along the stored sequence. The overview places the same selection on a coordinate axis. [Figure export in the desktop](../sequence_workspace.md#export-a-figure)

In [ ]:
linear = linearMapSvg(session)
overview = overviewSvg(session)
display(SVG(linear), SVG(overview))

## Map a larger plasmid

The 6,759 bp Dryad GFP plasmid has overlapping annotations. This figure uses All features to show the complete annotation set. Rotate the view 65° counterclockwise to spread the short labels across both margins. The sequence coordinates stay fixed. [Deposited GenBank record](../../databases/plasmids/maps/pBH4-His-SUMO-GFP-LPETGG.gb)

In [ ]:
gfpPath = root / "databases/plasmids/maps/pBH4-His-SUMO-GFP-LPETGG.gb"
gfp = readSequence(gfpPath).records[0]
gfpMap = plasmidMapSvg(DocumentSession(gfp), displayName=gfpPath.stem, rotationDeg=-65, width=1000, height=660, mapFeatureTypes=None)
display(SVG(gfpMap))

## Save the figures

Each run writes four standalone SVGs. Open either circular map at full size to inspect the labels.

In [ ]:
runId = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = root / "docs/notebooks/_outputs/V06" / runId
output.mkdir(parents=True)
figures = {"pUC19.svg": circular, "pUC19-linear.svg": linear, "pUC19-overview.svg": overview,
           gfpPath.with_suffix(".svg").name: gfpMap}
for name, svg in figures.items():
    (output / name).write_text(svg, encoding="utf-8")
print(output.relative_to(root))

## Compare translations at the same coordinates

In this short synthetic example, tables 1, 4 and 12 interpret the same four codons differently. The library maps each translated residue back to its bases. The printed coordinates are the centers of the codons, numbered from one.

In [ ]:
example = readPastedText("ATGTGACTGTAA", moleculeType="DNA").records[0]
for tableId in (1, 4, 12):
    frame = translateFrame(example, 1, GeneticCodePreference(tableId))
    centers = [codon.centerPosition + 1 for codon in frame.codons]
    print(f"Table {tableId:>2}   {frame.proteinSequence}   codon centers {centers}")